
# PyBlastAfterglowMag：Gaussian Jet 数据模拟（1000组，稳定版）

这个 notebook 按照你原来的 VegasAfterglow notebook 逻辑改写，并在原版基础上做了两类关键修正：

- **动力学更稳定**：收紧参数范围、加入更严格的 Gaussian jet 几何约束、缩短演化到极晚时间的推进范围。
- **读取更稳健**：不再假设请求频率一定都存在于输出文件里，而是优先读取 PyBlast 实际生成的 light-curve 频率；若版本差异导致无法直接读取频率网格，则自动跳过超出上限的频率点。

## 逻辑结构

1. 定义参数空间
2. 用 LHS 采样 1000 组物理参数
3. 逐组构造 PyBlastAfterglowMag 的 Gaussian jet + ISM 配置
4. 运行前向模拟并提取 light curve
5. 保存成长表数据集，方便后续做 surrogate / transfer learning

## 当前版本的设计原则

1. 默认采用 **Gaussian jet + ISM**。
2. 默认使用更稳妥的频率上限 `NU_MAX = 1e15`，这已经覆盖 radio 到 optical/near-UV，适合大多数 GRB afterglow surrogate 数据集。
3. 如果某次运行实际输出的最大频率低于你请求的上限，代码会自动裁剪到可用频率，避免再出现 `requested freqs > dfile unique freqs.max()`。
4. `PATH_TO_CPP` 仍然需要改成你本机真实的 `pba.out` 路径。


In [1]:

import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from dataclasses import dataclass
from enum import Enum
from tqdm.auto import tqdm
from typing import Optional

from PyBlastAfterglowMag.wrappers import run_grb
from PyBlastAfterglowMag.utils import cgs


/mnt/data/s2119250005/anaconda3/envs/GRB/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 一、全局配置

In [2]:

# ====== 数据规模 ======
N_THETA = 100   # 物理参数组数
K_TIME  = 48     # 每组参数保留的时间点数
M_FREQ  = 4      # 每组参数保留的频率点数（若输出上限不足，实际可能略少）

# ====== 全局时间范围（秒）=====
# 先使用更稳健的范围；如后续需要更晚期行为，可逐步往外扩展
T_MIN = 3e3
T_MAX = 1e9

# ====== 全局频率范围（Hz）=====
# 1e15 已覆盖 radio -> optical/near-UV，是一个适合作为 surrogate 第一版数据集的安全上限
NU_MIN = 1e9
NU_MAX = 1e15

# ====== 红移范围 ======
Z_MIN = 0.01
Z_MAX = 1.0

# ====== 结构喷流附加参数范围 ======
THETA_W_MIN = np.deg2rad(10.0)
THETA_W_MAX = np.deg2rad(25.0)

# ====== 数值设置 ======
RANDOM_SEED = 20260308
FLOOR_FLUX = 1e-300
RTOL = 5e-7
TB0, TB1, NTB = 3e2, 1e12, 600
N_LAYERS_A = 21

# ====== 路径设置 ======
WORK_ROOT = Path('./pyblast_runs_gaussian_1000')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# !!! 必改：改成你本机的 pba.out 路径 !!!
PATH_TO_CPP = '/mnt/data/s2119250005/PyBlastAfterglowMag/src/pba.out'

# 输出文件名
OUT_PARQUET = 'pyblast_gaussian_point_dataset_1000.parquet'
OUT_CSV = 'pyblast_gaussian_point_dataset_1000.csv'
OUT_THETA_CSV = 'pyblast_gaussian_theta_table_1000.csv'

rng = np.random.default_rng(RANDOM_SEED)


## 二、定义参数（log/linear 混合）+ LHS 采样工具

In [3]:

class Scale(Enum):
    LINEAR = 'linear'
    LOG = 'log'

@dataclass(frozen=True)
class ParamSpec:
    name: str
    low: float
    high: float
    scale: Scale

PARAM_SPECS = [
    ParamSpec('z',       Z_MIN,   Z_MAX,   Scale.LINEAR),
    ParamSpec('theta_v', 0.0,     0.35,    Scale.LINEAR),
    ParamSpec('Gamma0',  80.0,    400.0,   Scale.LOG),
    ParamSpec('E_iso',   1e51,    3e54,    Scale.LOG),
    ParamSpec('theta_c', 0.03,    0.20,    Scale.LOG),
    ParamSpec('theta_w', THETA_W_MIN, THETA_W_MAX, Scale.LINEAR),
    ParamSpec('n_ism',   1e-4,    1e-1,    Scale.LOG),
    ParamSpec('p',       2.05,    2.4,     Scale.LINEAR),
    ParamSpec('eps_e',   1e-4,    1e-1,    Scale.LOG),
    ParamSpec('eps_B',   1e-6,    1e-3,    Scale.LOG),
]

def sample_lhs_unit(n: int, d: int, seed: int) -> np.ndarray:
    try:
        from scipy.stats import qmc
        sampler = qmc.LatinHypercube(d=d, seed=seed)
        return sampler.random(n=n)
    except ImportError:
        print('WARNING: scipy 未安装，LHS 将退化为普通 uniform 采样。建议安装 scipy。')
        r = np.random.default_rng(seed)
        return r.random((n, d))

def transform_unit_to_param(u: np.ndarray, spec: ParamSpec) -> np.ndarray:
    if spec.scale == Scale.LINEAR:
        return spec.low + u * (spec.high - spec.low)
    if spec.scale == Scale.LOG:
        lo, hi = np.log10(spec.low), np.log10(spec.high)
        return 10 ** (lo + u * (hi - lo))
    raise ValueError(f'Unknown scale: {spec.scale}')


## 三、物理约束 + 采样参数表（1000组）

In [4]:

def apply_physical_constraints(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # core angle 限制在更稳定区间
    df['theta_c'] = np.clip(df['theta_c'].values, 0.03, 0.20)

    # wing angle 必须显著大于 core，且不要过宽
    df['theta_w'] = np.maximum(df['theta_w'].values, 2.5 * df['theta_c'].values)
    df['theta_w'] = np.clip(df['theta_w'].values, 0.18, 0.45)

    # observer angle 不要卡在外边界上，同时避免过强离轴
    df['theta_v'] = np.minimum(df['theta_v'].values, 0.8 * df['theta_w'].values)
    df['theta_v'] = np.minimum(df['theta_v'].values, 6.0 * df['theta_c'].values)
    df['theta_v'] = np.maximum(df['theta_v'].values, 0.0)

    # 微物理参数再做一次安全裁剪，避免边界误差
    df['eps_e'] = np.clip(df['eps_e'].values, 1e-4, 1e-1)
    df['eps_B'] = np.clip(df['eps_B'].values, 1e-6, 1e-3)
    df['p'] = np.clip(df['p'].values, 2.05, 2.4)

    return df

def sample_parameter_table(n_theta: int, specs: list[ParamSpec], seed: int) -> pd.DataFrame:
    d = len(specs)
    u = sample_lhs_unit(n=n_theta, d=d, seed=seed)

    data = {}
    for j, spec in enumerate(specs):
        data[spec.name] = transform_unit_to_param(u[:, j], spec)

    df = pd.DataFrame(data)
    df = apply_physical_constraints(df)
    df.insert(0, 'theta_id', np.arange(len(df), dtype=int))
    return df

theta_df = sample_parameter_table(N_THETA, PARAM_SPECS, seed=RANDOM_SEED)
theta_df.head()


,theta_id,z,theta_v,Gamma0,E_iso,theta_c,theta_w,n_ism,p,eps_e,eps_B
0,0,0.606180,0.009632,335.963444,4.602412e+52,0.091342,0.245159,0.012353,2.199829,0.000138,0.000016
1,1,0.756995,0.281945,99.095135,2.519462e+54,0.046991,0.414344,0.000591,2.132514,0.001173,0.000576
2,2,0.163464,0.208688,96.619795,3.891776e+51,0.057531,0.405380,0.002622,2.331204,0.001069,0.000060
3,3,0.120560,0.035112,230.955485,1.811160e+51,0.079982,0.300932,0.003312,2.094030,0.003767,0.000006
4,4,0.911386,0.034599,271.225097,9.950951e+52,0.110531,0.276327,0.000374,2.293507,0.001672,0.000004


## 四、z → 光度距离 d_L（cm）

In [5]:

def luminosity_distance_cm(z: float) -> float:
    try:
        from astropy.cosmology import Planck18 as cosmo
        return cosmo.luminosity_distance(z).to('cm').value
    except Exception:
        pass

    try:
        from scipy.integrate import quad
    except ImportError as e:
        raise ImportError(
            '需要 astropy 或 scipy 才能从 z 计算光度距离。建议：pip install astropy 或 pip install scipy'
        ) from e

    c_km_s = 299792.458
    H0 = 67.66
    Om0 = 0.3111
    Ol0 = 1.0 - Om0
    Mpc_to_cm = 3.085677581e24

    def E(zp):
        return np.sqrt(Om0 * (1 + zp) ** 3 + Ol0)

    integral, _ = quad(lambda zp: 1.0 / E(zp), 0, z, limit=200)
    d_c_mpc = (c_km_s / H0) * integral
    d_l_mpc = (1 + z) * d_c_mpc
    return d_l_mpc * Mpc_to_cm


## 五、PyBlast 配置构造 + 单样本模拟

In [6]:

def build_pba_config_from_row(row: pd.Series,
                              k_time: int,
                              m_freq: int,
                              t_min: float,
                              t_max: float,
                              nu_min: float,
                              nu_max: float) -> dict:
    z = float(row['z'])
    d_l = luminosity_distance_cm(z)

    main_pars = dict(
        d_l=d_l,
        z=z,
        n_ism=float(row['n_ism']),
        theta_obs=float(row['theta_v']),
        rtol=RTOL,
        lc_freqs=f'array logspace {nu_min} {nu_max} {m_freq}',
        lc_times=f'array logspace {t_min} {t_max} {k_time}',
        tb0=TB0,
        tb1=TB1,
        ntb=NTB,
    )

    struct = dict(
        struct='gaussian',
        Eiso_c=float(row['E_iso']),
        Gamma0c=float(row['Gamma0']),
        M0c=-1.0,
        theta_c=float(row['theta_c']),
        theta_w=float(row['theta_w']),
        n_layers_a=N_LAYERS_A,
    )

    grb_pars = dict(
        structure=struct,
        eps_e_fs=float(row['eps_e']),
        eps_b_fs=float(row['eps_B']),
        p_fs=float(row['p']),
        do_lc='yes',
        save_spec='no',
        method_synchrotron_fs='CSYN',
        method_ele_fs='numeric',
    )

    return {'main': main_pars, 'grb': grb_pars}


def _get_available_lc_freqs(ejecta) -> Optional[np.ndarray]:
    """尽量从当前 PyBlast 版本中读取实际可用的 LC 频率网格。"""
    candidate_methods = [
        'get_lc_freqs',
        'get_freqs',
        'get_lc_grid_freqs',
    ]
    for name in candidate_methods:
        if hasattr(ejecta, name):
            try:
                vals = np.asarray(getattr(ejecta, name)(), dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size > 0:
                    return np.unique(vals)
            except Exception:
                pass
    return None

def _choose_requested_freqs(ejecta, nu_min: float, nu_max: float, m_freq: int) -> np.ndarray:
    requested = np.logspace(np.log10(nu_min), np.log10(nu_max), m_freq)
    available = _get_available_lc_freqs(ejecta)

    # 优先使用输出文件里真实存在的频率，避免 requested > available.max 的读取错误
    if available is not None and available.size > 0:
        available = available[(available >= nu_min) & (available <= nu_max)]
        if available.size == 0:
            available = _get_available_lc_freqs(ejecta)
        if available is not None and available.size > 0:
            picked = []
            for nu in requested:
                idx = int(np.argmin(np.abs(np.log10(available) - np.log10(nu))))
                picked.append(float(available[idx]))
            picked = np.unique(np.asarray(picked, dtype=float))
            return picked

    # 若无法直接读取频率网格，则退回到原始请求；后面逐频读取时再自动跳过越界频率
    return requested


def _parse_available_fmax_from_error(msg: str) -> Optional[float]:
    m = re.search(r"freqs=.*? > dfile unique freqs\.max\(\)=([0-9eE+\-.]+)", msg)
    if m is None:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None

def run_single_simulation(row: pd.Series,
                          work_root: Path,
                          path_to_cpp: str,
                          k_time: int,
                          m_freq: int,
                          t_min: float,
                          t_max: float,
                          nu_min: float,
                          nu_max: float,
                          overwrite: bool = True) -> pd.DataFrame:
    theta_id = int(row['theta_id'])
    working_dir = work_root / f'run_{theta_id:04d}'

    if overwrite and working_dir.exists():
        shutil.rmtree(working_dir)
    working_dir.mkdir(parents=True, exist_ok=True)

    P = build_pba_config_from_row(
        row=row,
        k_time=k_time,
        m_freq=m_freq,
        t_min=t_min,
        t_max=t_max,
        nu_min=nu_min,
        nu_max=nu_max,
    )

    pba = run_grb(
        working_dir=str(working_dir) + '/',
        P=P,
        run=True,
        path_to_cpp=path_to_cpp,
        loglevel='err',
        process_skymaps=False,
    )

    ejecta = pba.GRB
    times = np.asarray(ejecta.get_lc_times(), dtype=float)
    freqs = _choose_requested_freqs(ejecta, nu_min=nu_min, nu_max=nu_max, m_freq=m_freq)

    chunks = []
    skipped = []
    for freq in freqs:
        try:
            lc = np.asarray(ejecta.get_lc(freq=float(freq)), dtype=float)
        except ValueError as e:
            msg = str(e)
            avail_fmax = _parse_available_fmax_from_error(msg)
            if avail_fmax is not None and float(freq) > avail_fmax:
                skipped.append((float(freq), avail_fmax))
                continue
            raise

        df_i = pd.DataFrame({
            'theta_id': theta_id,
            't_s': times,
            'nu_hz': float(freq),
            'F_nu_mJy': lc,
        })
        for col in row.index:
            if col != 'theta_id':
                df_i[col] = row[col]
        chunks.append(df_i)

    if len(chunks) == 0:
        extra = ''
        if len(skipped) > 0:
            extra = f' Requested freqs were all above available output range; first skipped={skipped[0]}.'
        raise RuntimeError(f'theta_id={theta_id} 没有成功读取到任何 light curve。{extra}')

    out = pd.concat(chunks, ignore_index=True)
    out['log10_t'] = np.log10(out['t_s'])
    out['log10_nu'] = np.log10(out['nu_hz'])

    finite = np.isfinite(out['F_nu_mJy'].values)
    positive = out['F_nu_mJy'].values > 0
    out['valid'] = finite & positive

    safe_flux = np.maximum(out['F_nu_mJy'].values, FLOOR_FLUX)
    out['log10_F_nu_mJy'] = np.where(out['valid'].values, np.log10(safe_flux), np.nan)
    out['n_freq_used'] = out.groupby('theta_id')['nu_hz'].transform('nunique')
    return out


## 六、批量生成点样本数据集

In [7]:

def generate_point_dataset(theta_df: pd.DataFrame,
                           work_root: Path,
                           path_to_cpp: str,
                           k_time: int,
                           m_freq: int,
                           t_min: float,
                           t_max: float,
                           nu_min: float,
                           nu_max: float,
                           overwrite_each_run: bool = True,
                           stop_on_error: bool = False) -> pd.DataFrame:
    chunks = []
    failures = []

    for _, row in tqdm(theta_df.iterrows(), total=len(theta_df)):
        try:
            df_i = run_single_simulation(
                row=row,
                work_root=work_root,
                path_to_cpp=path_to_cpp,
                k_time=k_time,
                m_freq=m_freq,
                t_min=t_min,
                t_max=t_max,
                nu_min=nu_min,
                nu_max=nu_max,
                overwrite=overwrite_each_run,
            )
            chunks.append(df_i)
        except Exception as e:
            print(f"[FAILED] theta_id={int(row['theta_id'])}")
            print('row =')
            print(row.to_dict())
            print('error =', repr(e))
            failures.append({
                'theta_id': int(row['theta_id']),
                'error': repr(e),
            })
            if stop_on_error:
                raise

    if len(chunks) == 0:
        raise RuntimeError('所有样本都运行失败，请检查 PATH_TO_CPP、PyBlast 安装和参数设置。')

    point_df = pd.concat(chunks, ignore_index=True)
    fail_df = pd.DataFrame(failures)
    return point_df, fail_df


## 七、先跑少量测试（强烈建议先执行这一格）

建议先用 1 组 smoke test 验证环境、参数范围和频率读取逻辑，再逐步放大到 10 组、100 组、1000 组。


In [8]:

# 先拿前 5 组测试，确认 PyBlast 配置、路径、输出都正常
TEST_N = 5

test_point_df, test_fail_df = generate_point_dataset(
    theta_df=theta_df.iloc[:TEST_N].copy(),
    work_root=WORK_ROOT / 'smoke_test',
    path_to_cpp=PATH_TO_CPP,
    k_time=K_TIME,
    m_freq=M_FREQ,
    t_min=T_MIN,
    t_max=T_MAX,
    nu_min=NU_MIN,
    nu_max=NU_MAX,
    overwrite_each_run=True,
    stop_on_error=False,
)

print('test_point_df.shape =', test_point_df.shape)
print('test_fail_df.shape  =', test_fail_df.shape)
display(test_fail_df.head())
test_point_df.head()


 20%|██        | 1/5 [05:27<21:48, 327.08s/it]ERROR   ] : [ blastwave.h:1527 ] : 

 mom=335.279 Gamma=335.28 beta=0.999996
 mom=329.863 Gamma=329.864 beta=0.999995
 mom=319.292 Gamma=319.294 beta=0.999995
 mom=304.07 Gamma=304.072 beta=0.999995
 mom=284.901 Gamma=284.903 beta=0.999994
 mom=262.635 Gamma=262.637 beta=0.999993
 mom=238.208 Gamma=238.21 beta=0.999991
 mom=212.578 Gamma=212.58 beta=0.999989
 mom=186.659 Gamma=186.661 beta=0.999986
 mom=161.275 Gamma=161.278 beta=0.999981
 mom=137.12 Gamma=137.123 beta=0.999973
 mom=114.732 Gamma=114.736 beta=0.999962
 mom=94.4853 Gamma=94.4906 beta=0.999944
 mom=76.5974 Gamma=76.6039 beta=0.999915
 mom=61.1405 Gamma=61.1487 beta=0.999866
 mom=48.0672 Gamma=48.0776 beta=0.999784
 mom=37.2366 Gamma=37.25 beta=0.99964
 mom=28.4429 Gamma=28.4604 beta=0.999383
 mom=21.4418 Gamma=21.4651 beta=0.998914
 mom=15.9733 Gamma=16.0046 beta=0.998046
 mom=11.7805 Gamma=11.8229 beta=0.996417
Wrong value in RHS: Gamma=-0.0295337 Gamma0=319.294 E0=1.34095e+49 M0=4.68751e+25 R=4.98023e+18 Eint2=159.249 theta=0.0814014 M2=250.408 rho=4.4103

 20%|██        | 1/5 [05:47<23:09, 347.28s/it]


KeyboardInterrupt: 

## 八、正式生成 1000 组数据

如果 smoke test 通过，建议先把 `theta_df.iloc[:10]`、`theta_df.iloc[:100]` 跑通，再执行整套 1000 组。


In [ ]:

# 确认上面的 smoke test 没问题后，再运行这一格。
point_df, fail_df = generate_point_dataset(
    theta_df=theta_df,
    work_root=WORK_ROOT,
    path_to_cpp=PATH_TO_CPP,
    k_time=K_TIME,
    m_freq=M_FREQ,
    t_min=T_MIN,
    t_max=T_MAX,
    nu_min=NU_MIN,
    nu_max=NU_MAX,
    overwrite_each_run=True,
    stop_on_error=False,
)

print('point_df.shape =', point_df.shape)
print('fail_df.shape  =', fail_df.shape)
point_df.head()


## 九、保存数据

In [ ]:

theta_df.to_csv(OUT_THETA_CSV, index=False)
print(f'Saved: {OUT_THETA_CSV}  rows={len(theta_df)}')

try:
    point_df.to_parquet(OUT_PARQUET, index=False)
    print(f'Saved: {OUT_PARQUET}  rows={len(point_df)}')
except Exception as e:
    print('Parquet 保存失败（可能缺 pyarrow/fastparquet），改存 CSV。错误：', repr(e))
    point_df.to_csv(OUT_CSV, index=False)
    print(f'Saved: {OUT_CSV}  rows={len(point_df)}')

if 'fail_df' in globals() and len(fail_df) > 0:
    fail_df.to_csv('pyblast_gaussian_failures_1000.csv', index=False)
    print('Saved: pyblast_gaussian_failures_1000.csv')


## 十、快速检查一组 light curve

In [ ]:

example_theta_id = int(point_df['theta_id'].iloc[0])
sub = point_df[point_df['theta_id'] == example_theta_id].copy()

fig, ax = plt.subplots(figsize=(6, 4))
for freq, grp in sub.groupby('nu_hz'):
    grp = grp.sort_values('t_s')
    ax.plot(grp['t_s'] / cgs.day, grp['F_nu_mJy'], label=f'{freq:.2e} Hz')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('time [day]')
ax.set_ylabel('Flux density [mJy]')
ax.set_title(f'theta_id = {example_theta_id}')
ax.grid(ls=':')
ax.legend(fontsize=8)
plt.show()
